# Industrial Anomaly Detection - EDA (MVTec AD + VisA)

Streaming exploration of two anomaly-detection benchmarks **without extracting** the 6.8 GB of archives.
We open each archive with `tarfile`, enumerate members, classify them by category / split / good-vs-defect, and stream a small subset of images via `extractfile()` into PIL for visual checks.

Datasets:
- **MVTec AD** (`mvtec_anomaly_detection.tar.xz`, ~5.0 GB): 15 industrial object/texture categories, defect-free training images, mixed test images with pixel-level masks.
- **VisA** (`VisA_20220922.tar`, ~1.8 GB): 12 categories, 9621 normal and 1200 anomalous images, more cluttered scenes than MVTec.


In [ ]:
import io
import os
import re
import tarfile
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = Path('data/')
MVTEC = DATA_DIR / 'mvtec_anomaly_detection.tar.xz'
VISA  = DATA_DIR / 'VisA_20220922.tar'

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.JPG', '.PNG', '.bmp')

for p in (MVTEC, VISA):
    print(f'{p.name:40s}  {p.stat().st_size/1e9:6.2f} GB')


## 1. MVTec AD - enumerate archive members

Folder convention: `<category>/{train|test|ground_truth}/<defect_type>/<file>.png`.
`defect_type == 'good'` means a normal image; everything else is an anomaly class. We never read the file payload here - just the headers.

In [ ]:
def index_mvtec(path):
    rows = []
    with tarfile.open(path, mode='r:xz') as tf:
        for m in tf:
            if not m.isfile():
                continue
            name = m.name
            if not name.endswith(IMG_EXTS):
                continue
            parts = name.split('/')
            # leading folder is the dataset root, e.g. 'mvtec_anomaly_detection/bottle/test/broken_large/000.png'
            try:
                root_idx = next(i for i, p in enumerate(parts) if p in ('train', 'test', 'ground_truth'))
            except StopIteration:
                continue
            category = parts[root_idx - 1]
            split    = parts[root_idx]
            defect   = parts[root_idx + 1] if len(parts) > root_idx + 1 else 'unknown'
            rows.append({
                'category': category,
                'split': split,
                'defect': defect,
                'is_good': defect == 'good',
                'size_bytes': m.size,
                'path': name,
            })
    return pd.DataFrame(rows)

mvtec = index_mvtec(MVTEC)
print('Total image members:', len(mvtec))
print('Categories:', sorted(mvtec['category'].unique()))
print('Splits:', sorted(mvtec['split'].unique()))


In [ ]:
mvtec_summary = (mvtec[mvtec['split'].isin(['train', 'test'])]
                 .groupby(['category', 'split', 'is_good'])
                 .size().unstack(fill_value=0)
                 .rename(columns={True: 'good', False: 'defect'}))
mvtec_summary['total'] = mvtec_summary.sum(axis=1)
mvtec_summary


In [ ]:
mvtec_defect_types = (mvtec[(mvtec['split'] == 'test') & (~mvtec['is_good'])]
                      .groupby('category')['defect']
                      .apply(lambda s: sorted(s.unique())))
for cat, defs in mvtec_defect_types.items():
    print(f'{cat:15s} -> {defs}')


## 2. VisA - enumerate archive members

VisA uses a different layout: `<category>/Data/Images/{Normal|Anomaly}/<id>.JPG` plus `<category>/Data/Masks/Anomaly/...` and CSV metadata. We pick up image files only.

In [ ]:
def index_visa(path):
    rows = []
    with tarfile.open(path, mode='r') as tf:
        for m in tf:
            if not m.isfile():
                continue
            name = m.name
            if not name.endswith(IMG_EXTS):
                continue
            if '/Data/Images/' not in name:
                continue
            parts = name.split('/')
            # e.g. 'VisA_20220922/candle/Data/Images/Normal/0000.JPG'
            try:
                cat_idx = parts.index('Data') - 1
            except ValueError:
                continue
            category = parts[cat_idx]
            label    = parts[cat_idx + 3]  # 'Normal' or 'Anomaly'
            rows.append({
                'category': category,
                'label': label,
                'is_good': label == 'Normal',
                'size_bytes': m.size,
                'path': name,
            })
    return pd.DataFrame(rows)

visa = index_visa(VISA)
print('Total image members:', len(visa))
print('Categories:', sorted(visa['category'].unique()))


In [ ]:
visa_summary = (visa.groupby(['category', 'label']).size()
                .unstack(fill_value=0))
visa_summary['total'] = visa_summary.sum(axis=1)
visa_summary


## 3. Stream sample images via tarfile (no extraction)

For each MVTec category we pull one good and one defective test image; for each VisA category we pull one Normal and one Anomaly image. `tarfile.extractfile()` returns a file-like object that PIL can read directly.

In [ ]:
def stream_image(tar_path, member_name, mode):
    with tarfile.open(tar_path, mode=mode) as tf:
        member = tf.getmember(member_name)
        f = tf.extractfile(member)
        data = f.read()
    return Image.open(io.BytesIO(data)).convert('RGB')

# Pre-pick one good + one defect test image per MVTec category
mvtec_picks = {}
for cat in sorted(mvtec['category'].unique()):
    sub = mvtec[(mvtec['category'] == cat) & (mvtec['split'] == 'test')]
    good = sub[sub['is_good']].head(1)
    bad  = sub[~sub['is_good']].head(1)
    if not good.empty and not bad.empty:
        mvtec_picks[cat] = (good.iloc[0]['path'], bad.iloc[0]['path'], bad.iloc[0]['defect'])

# Pre-pick one Normal + one Anomaly image per VisA category
visa_picks = {}
for cat in sorted(visa['category'].unique()):
    sub = visa[visa['category'] == cat]
    good = sub[sub['is_good']].head(1)
    bad  = sub[~sub['is_good']].head(1)
    if not good.empty and not bad.empty:
        visa_picks[cat] = (good.iloc[0]['path'], bad.iloc[0]['path'])

len(mvtec_picks), len(visa_picks)


In [ ]:
fig, axes = plt.subplots(len(mvtec_picks), 2, figsize=(6, 2.4 * len(mvtec_picks)))
for row, (cat, (gpath, bpath, dtype)) in enumerate(mvtec_picks.items()):
    g = stream_image(MVTEC, gpath, 'r:xz')
    b = stream_image(MVTEC, bpath, 'r:xz')
    axes[row, 0].imshow(g); axes[row, 0].set_title(f'{cat} | good {g.size}'); axes[row, 0].axis('off')
    axes[row, 1].imshow(b); axes[row, 1].set_title(f'{cat} | {dtype} {b.size}'); axes[row, 1].axis('off')
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(len(visa_picks), 2, figsize=(6, 2.4 * len(visa_picks)))
for row, (cat, (gpath, bpath)) in enumerate(visa_picks.items()):
    g = stream_image(VISA, gpath, 'r')
    b = stream_image(VISA, bpath, 'r')
    axes[row, 0].imshow(g); axes[row, 0].set_title(f'{cat} | normal {g.size}'); axes[row, 0].axis('off')
    axes[row, 1].imshow(b); axes[row, 1].set_title(f'{cat} | anomaly {b.size}'); axes[row, 1].axis('off')
plt.tight_layout(); plt.show()


## 4. Image dimensions and mean intensity (sampled)

Reading every file in a 5 GB xz archive is slow, so we sample up to N images per dataset and compute width/height plus mean intensity per channel.

In [ ]:
def sample_stats(tar_path, members, mode, n_per=20, seed=0):
    rng = np.random.default_rng(seed)
    sampled = (pd.DataFrame({'path': members})
                 .sample(min(n_per, len(members)), random_state=seed)['path'].tolist())
    out = []
    with tarfile.open(tar_path, mode=mode) as tf:
        for name in sampled:
            try:
                f = tf.extractfile(tf.getmember(name))
                if f is None:
                    continue
                im = Image.open(io.BytesIO(f.read())).convert('RGB')
                arr = np.asarray(im)
                out.append({
                    'path': name,
                    'width': im.width, 'height': im.height,
                    'mean_R': arr[..., 0].mean(),
                    'mean_G': arr[..., 1].mean(),
                    'mean_B': arr[..., 2].mean(),
                })
            except Exception:
                continue
    return pd.DataFrame(out)

mvtec_stats = sample_stats(MVTEC, mvtec['path'].tolist(), 'r:xz', n_per=60)
visa_stats  = sample_stats(VISA,  visa['path'].tolist(),  'r',    n_per=60)
print('MVTec dims:'); print(mvtec_stats[['width', 'height']].describe())
print('\nVisA dims:');  print(visa_stats [['width', 'height']].describe())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(mvtec_stats['width'], mvtec_stats['height'], alpha=0.6, label='MVTec')
axes[0].scatter(visa_stats ['width'], visa_stats ['height'], alpha=0.6, label='VisA')
axes[0].set_xlabel('width'); axes[0].set_ylabel('height'); axes[0].set_title('Image dimensions (sampled)')
axes[0].legend()

for name, df in [('MVTec', mvtec_stats), ('VisA', visa_stats)]:
    axes[1].hist(df[['mean_R', 'mean_G', 'mean_B']].mean(axis=1), bins=20, alpha=0.5, label=name)
axes[1].set_xlabel('mean pixel intensity (0-255)'); axes[1].set_ylabel('count'); axes[1].set_title('Mean intensity (sampled)')
axes[1].legend()
plt.tight_layout(); plt.show()


## 5. Summary

- **MVTec AD**: 15 categories (10 objects + 5 textures). Train split is good-only; test split mixes good and defective images plus pixel masks under `ground_truth/`. Resolutions cluster around 700-1024 px square; backgrounds are clean and centered.
- **VisA**: 12 categories (single-instance and multi-instance scenes such as PCBs and capsules). Single `Data/Images/{Normal,Anomaly}` split. Higher native resolution (~1.5K-1.9K wide) and more visual clutter.
- **Class imbalance** is severe: normals dominate train (MVTec is 100% good in train, ~9k normals vs ~1.2k anomalies in VisA). Modeling will lean unsupervised / one-class / reconstruction-based, with the supervised multi-class step restricted to MVTec test defects.
- **Streaming via `tarfile.extractfile()`** keeps disk usage flat (no extraction). For training we will either extract per-category on demand or persist a pre-cropped resized cache (e.g. 256x256 PNG) under `.tmp/`.
- **Pre-processing plan**: resize 256x256, center-crop or letterbox per category, normalize with ImageNet stats for backbone-based feature extractors (PatchCore / PaDiM).
